# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [7]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (0, 0)
        self.walls = {(1,1),(0,3),(2,4),(4,2)}
        self.slippery_states = {(1,2),(2,1),(3,3)}

        self.terminal_states = { 
            (0, 5): 10,
            (3, 5): -10,
            (2, 2): 2
        }

        self.danger_states = {
            (1, 4): -3,
            (4, 1): -3
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if row < 0 or row >= self.height:
            return False
        if col < 0 or col >= self.width:
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        for row in range(self.height):
            for col in range(self.width):
                state = (row, col)
                if self.is_valid_state(state):
                    yield state

    def is_terminal(self, state):
        return state in self.terminal_states or state in self.danger_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):

        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        row, col = state
        delta_row, delta_col = action
        next_state = (row + delta_row, col + delta_col)

        if not self.is_valid_state(next_state):
            next_state = state

        if state in self.slippery_states:
            # Resbaloso: 80% hacia la acción deseada, 10% a cada lado
            left_action = (-delta_col, delta_row)  # Acción a la izquierda
            right_action = (delta_col, -delta_row)  # Acción a la derecha

            left_state = (row + left_action[0], col + left_action[1])
            right_state = (row + right_action[0], col + right_action[1])

            if not self.is_valid_state(left_state):
                left_state = state
            if not self.is_valid_state(right_state):
                right_state = state

            return [
                (next_state, 0.8),
                (left_state, 0.1),
                (right_state, 0.1)
            ]
        else:
            # No resbaloso: 100% hacia la acción deseada
            return [(next_state, 1.0)]



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [9]:
grid = WarehouseMDP()

S = list(grid.states())
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [14]:
def expected_next_value(grid, state, action, V):
    """
    Calcula el valor esperado de la siguiente acción:
        sum_{s'} T(s,a,s') V(s')

    Args:
        grid: instancia de WarehouseMDP
        state: estado actual (row, col)
        action: acción a tomar (delta_row, delta_col)
        V: diccionario de valores {state: value}

    Returns:
        float: valor esperado
    """
    transitions = grid.get_transition_probs(state, action)
    expected_value = 0.0
    for next_state, prob in transitions:
        expected_value += prob * V[next_state]
    return expected_value


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    """
    Implementa el algoritmo de iteración de valores para encontrar V*.

    Args:
        grid: instancia de WarehouseMDP
        threshold: umbral de convergencia
        max_iter: número máximo de iteraciones

    Returns:
        V: diccionario de valores {state: value}
    """
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        delta = 0.0
        new_V = V.copy()

        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                action_values = []
                for a in grid.actions:
                    expected_value = expected_next_value(grid, s, a, V)
                    action_values.append(expected_value)
                new_V[s] = grid.get_reward(s) + grid.gamma * max(action_values)

            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V

        if delta < threshold:
            print(f"Convergió en {iteration + 1} iteraciones.")
            break
    else:
        print("No convergió dentro del número máximo de iteraciones.")

    return V, iteration


def extract_policy(grid, V):
    """
    Extrae la política óptima pi* a partir de V*.

    Args:
        grid: instancia de WarehouseMDP
        V: diccionario de valores {state: value}

    Returns:
        pi: diccionario de políticas {state: action}
    """
    pi = {}
    for s in grid.states():
        if grid.is_terminal(s):
            pi[s] = None  # No hay acción en estados terminales
        else:
            best_action = None
            best_value = float('-inf')
            for a in grid.actions:
                expected_value = expected_next_value(grid, s, a, V)
                if expected_value > best_value:
                    best_value = expected_value
                    best_action = a
            pi[s] = best_action
    return pi



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [20]:
from random import random


def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    """
    Evalúa la política actual para encontrar V^pi.

    Args:
        grid: instancia de WarehouseMDP
        policy: diccionario de políticas {state: action}
        threshold: umbral de convergencia
        max_iter: número máximo de iteraciones

    Returns:
        V: diccionario de valores {state: value}
    """
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(max_iter):
        delta = 0.0
        new_V = V.copy()

        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                a = policy[s]
                expected_value = expected_next_value(grid, s, a, V)
                new_V[s] = grid.get_reward(s) + grid.gamma * expected_value

            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V

        if delta < threshold:
            print(f"Convergió en {iteration + 1} iteraciones.")
            break
    else:
        print("No convergió dentro del número máximo de iteraciones.")

    return V

def policy_improvement(grid, V):
    """
    Mejora la política actual a partir de V.

    Args:
        grid: instancia de WarehouseMDP
        V: diccionario de valores {state: value}

    Returns:
        pi: diccionario de políticas {state: action}
    """
    pi = {}
    for s in grid.states():
        if grid.is_terminal(s):
            pi[s] = None  # No hay acción en estados terminales
        else:
            best_action = None
            best_value = float('-inf')
            for a in grid.actions:
                expected_value = expected_next_value(grid, s, a, V)
                if expected_value > best_value:
                    best_value = expected_value
                    best_action = a
            pi[s] = best_action
    return pi


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    """
    Implementa el algoritmo de iteración de políticas para encontrar pi*.

    Args:
        grid: instancia de WarehouseMDP
        threshold: umbral de convergencia
        max_iter: número máximo de iteraciones

    Returns:
        pi: diccionario de políticas {state: action}
        V: diccionario de valores {state: value}
        History: lista de tuplas (iteration, V) para cada iteración
    """
    # 1. Inicializar una política aleatoria
    import random

    # Reemplaza la línea del error por esta:
    pi = {s: random.choice(grid.actions) for s in grid.states() if not grid.is_terminal(s)}
    V = {s: 0.0 for s in grid.states()}
    History = []

    for iteration in range(max_iter):
        # 2. Evaluar la política actual
        V = policy_evaluation(grid, pi, threshold, max_iter)

        # 3. Mejorar la política
        new_pi = policy_improvement(grid, V)
        History.append((iteration, V.copy()))
        # 4. Verificar si la política ha cambiado
        if new_pi == pi:
            print(f"Política establecida en {iteration + 1} iteraciones.")
            break

        pi = new_pi
    else:
        print("No convergió dentro del número máximo de iteraciones.")

    return pi, V, History



## Parte 4 — Visualización y comparación


In [12]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [21]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


Convergió en 13 iteraciones.
=== VALUE ITERATION ===
Iteraciones: 12

Valores:
 -2.378 |  -1.531 |  -0.590 |   WALL   |  +8.000 | +10.000
 -1.531 |   WALL   |  +0.456 |  -0.280 |  -3.000 |  +8.000
 -0.590 |  +0.456 |  +2.000 |  +0.800 |   WALL   |  +6.200
 -1.252 |  -0.280 |  +0.800 |  -0.481 |  -1.433 | -10.000
 -2.127 |  -3.000 |   WALL   |  -1.433 |  -2.290 |  -3.061

Política:
 ↓  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  ↓  | -3 |  ↑ 
 →  |  →  | +2 |  ←  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  | -3 |  #  |  ↑  |  ↑  |  ← 
Convergió en 89 iteraciones.
Convergió en 89 iteraciones.
Convergió en 89 iteraciones.
Convergió en 13 iteraciones.
Convergió en 13 iteraciones.
Política establecida en 5 iteraciones.

=== POLICY ITERATION ===
Historia: [(0, {(0, 0): -9.999153585021714, (0, 1): -9.999153585021714, (0, 2): -9.999153585021714, (0, 4): -9.999153585021714, (0, 5): 10, (1, 0): -9.999153585021714, (1, 2): 0.11758241758241761, (1, 3): -3.7, (1, 4): -3, (1, 5): -9.99915


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
Según la política y los valores impresos, el robot prefiere dirigirse primero a la estación de carga +2. Desde START = (0, 0), la política selecciona bajar; luego conduce hacia la derecha por la fila inferior de la zona izquierda hasta llegar a (2, 2), donde obtiene +2. Aunque la entrega ofrece una recompensa inmediata mayor (+10), llegar a ella implica una ruta más larga y pasar cerca del peligro -3. Además, cada paso tiene una recompensa de -1, por lo que los trayectos largos acumulan costes. La política maximiza la recompensa descontada esperada, no simplemente la recompensa terminal más alta.

Hay un matiz: desde START, bajar y moverse a la derecha tienen el mismo valor esperado en los resultados mostrados por la simetría del mapa; la función extract_policy() elige bajar porque esa acción aparece antes que derecha en la lista de acciones. La ruta que sigue después de ese desempate conduce hacia la estación +2.

2. ¿Por qué una recompensa menor podría ser óptima?
Porque la decisión depende de la suma esperada de recompensas futuras descontadas, incluyendo el coste por cada paso y los riesgos de la trayectoria, no solo de la recompensa final. Una recompensa de +2 alcanzada en pocos pasos y con menor exposición al peligro puede producir un retorno esperado mayor que una recompensa de +10 que exige un recorrido largo o arriesgado. En este caso, la recompensa de +10 no compensa suficientemente los costes y riesgos adicionales según el modelo implementado.

3. ¿En qué estados el piso resbaloso cambia la decisión?
El piso resbaloso modifica la evaluación de las acciones en los estados (1, 2), (2, 1) y (3, 3), porque el movimiento elegido deja de ser seguro y aparecen probabilidades de desviación. Esto puede hacer preferible una acción que, aunque parezca menos directa, tenga vecinos más favorables o evite caer en un estado de baja recompensa. En la política mostrada, la acción en (2, 1) es moverse a la derecha hacia la estación +2; en (1, 2) se elige bajar, y en (3, 3) se elige subir. No obstante, para atribuir cuantitativamente esas decisiones al nivel de resbalamiento, hay que corregir primero las probabilidades del código, pues las que se ejecutaron no son las que especifica el enunciado.

4. ¿Qué papel cumple el costo por paso `-1`?
El -1 representa el coste de seguir operando y hace que cada movimiento adicional reduzca el retorno acumulado. Por eso el agente tiene un incentivo para llegar a una recompensa útil en pocos pasos y evitar recorridos innecesarios. También evita que las recompensas positivas lejanas se consideren automáticamente mejores que las recompensas más pequeñas y cercanas. Su efecto depende de gamma: las recompensas futuras se descuentan y, por tanto, su importancia relativa disminuye con la distancia temporal.

5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.
